In [1]:
import sys
import os 
import torch 
import numpy as np 
for stream in (sys.stdout, sys.stderr):
    if stream is not None and not hasattr(stream, "reconfigure"):
        stream.reconfigure = lambda *args, **kwargs: None

from process_uv_maps_architecture import BioSkinPipeline, LDMEvaluator, PARAM_NAMES 


import csv
import json
import torch.nn as nn
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

CHECKPOINT    = r"D:\Github\PhD Code\Biophysical-LDM\Pretrain_Model\BioSkinAO.pt"


import torch
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))

0 NVIDIA GeForce RTX 3090 (8, 6)
1 NVIDIA GeForce RTX 5070 Ti (12, 0)


c:\Users\Aai\anaconda3\envs\bioskin\lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5070 Ti with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5070 Ti GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════════

PARAM_NAMES = [
    'melanin',
    'hemoglobin',
    'epidermal_thickness',
    'eumelanin_ratio',
    'oxygenation',
]

PARAM_NOTATION = {
    'melanin':             'm',
    'hemoglobin':          'h',
    'epidermal_thickness': 'e',
    'eumelanin_ratio':     'r',
    'oxygenation':         'o',
}

PARAMS_WITH_STD = {'melanin', 'hemoglobin'}

PARAM_COLORMAPS = {
    'melanin':             'YlOrBr',
    'hemoglobin':          'Reds',
    'epidermal_thickness': 'Blues',
    'eumelanin_ratio':     'Oranges',
    'oxygenation':         'RdYlGn_r',
}

# Default exposure value — BioSkin was trained with exposure-aware input.
# Mean exposure (0.5) is a safe neutral default for inference.
DEFAULT_EXPOSURE = 0.5



In [4]:
ALBEDO_PATH   = r"D:\Github\PhD Code\Biophysical-LDM\dataset\Albedo-UV\000011.png"


In [17]:
# Load checkpoint and inspect encoder/decoder weights
cktp = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
print("Model loaded successfully.")
print(cktp.keys())

# Build weight dictionaries from the checkpoint before using them
enc_weights = {
    k.replace("module.", ""): v
    for k, v in cktp.items()
    if "fc_enc" in k
}

dec_weights = {
    k.replace("module.", ""): v
    for k, v in cktp.items()
    if "fc_dec" in k
}

print("Encoder weights:")
for k, v in enc_weights.items():
    print(k, tuple(v.shape))

print("Decoder weights:")
for k, v in dec_weights.items():
    print(k, tuple(v.shape))

Model loaded successfully.
odict_keys(['module.fc_enc_in.weight', 'module.fc_enc_in.bias', 'module.fc_enc.weight', 'module.fc_enc.bias', 'module.fc_enc_out.weight', 'module.fc_enc_out.bias', 'module.fc_dec_in.weight', 'module.fc_dec_in.bias', 'module.fc_dec.weight', 'module.fc_dec.bias', 'module.fc_dec_out.weight', 'module.fc_dec_out.bias'])
Encoder weights:
fc_enc_in.weight (70, 3)
fc_enc_in.bias (70,)
fc_enc.weight (70, 70)
fc_enc.bias (70,)
fc_enc_out.weight (6, 70)
fc_enc_out.bias (6,)
Decoder weights:
fc_dec_in.weight (512, 6)
fc_dec_in.bias (512,)
fc_dec.weight (512, 512)
fc_dec.bias (512,)
fc_dec_out.weight (310, 512)
fc_dec_out.bias (310,)


In [7]:
dec_weights = {
    k.replace("module.", ""): v
    for k, v in ckt.items()
    if "fc_dec" in k
}

print("Decoder weights loaded successfully.")
print(dec_weights.keys())
print(f"Decoder weights shape: {dec_weights['fc_dec.weight'].shape}")

Decoder weights loaded successfully.
dict_keys(['fc_dec_in.weight', 'fc_dec_in.bias', 'fc_dec.weight', 'fc_dec.bias', 'fc_dec_out.weight', 'fc_dec_out.bias'])
Decoder weights shape: torch.Size([512, 512])


In [18]:
class BioSkinEncoder(nn.Module):
    """
    BioSkinAO encoder built directly from the .pt checkpoint.

    Architecture (from BioSkinAO.json):
        Input:  RGB (3) + exposure (1) = 4  [exposure_aware: true]
        Hidden: 70 → 70                     [H_enc: 70, network_structure: 2]
        Output: 5 skin parameters           [D_skin: 5]
        Activation: ReLU
        Output space: log-space → apply exp() for physical values

    Args:
        checkpoint_path: Path to BioSkinAO.pt
        device:          'cuda' or 'cpu'
    """

    def __init__(self, checkpoint_path: str, device: str = 'cpu'):
        super().__init__()

        # ── Build architecture from JSON spec ─────────────────────────────────
        # exposure_aware=True → input is RGB(3) + exposure(1) = 4
        self.fc_enc_in  = nn.Linear(3,  70)
        self.fc_enc     = nn.Linear(70, 70)
        self.fc_enc_out = nn.Linear(70, 6)
        self.act        = nn.Tanh()
        self.act_final  = nn.Sigmoid()

        # ── Load weights ──────────────────────────────────────────────────────
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

        # Strip 'module.' prefix from DataParallel training
        enc_weights = {
            k.replace('module.', ''): v
            for k, v in ckpt.items()
            if 'fc_enc' in k
        }

        missing, unexpected = self.load_state_dict(enc_weights, strict=True)
        if missing:
            raise RuntimeError(f"Missing keys in checkpoint: {missing}")
        if unexpected:
            print(f"  [BioSkinEncoder] Unexpected keys (ignored): {unexpected}")

        self.eval()
        self.requires_grad_(False)
        self.to(device)
        self.device = device

        print(f"[BioSkinEncoder] ✓ Loaded on {device}")
        print(f"[BioSkinEncoder]   fc_enc_in:  {self.fc_enc_in.weight.shape}")
        print(f"[BioSkinEncoder]   fc_enc:     {self.fc_enc.weight.shape}")
        print(f"[BioSkinEncoder]   fc_enc_out: {self.fc_enc_out.weight.shape}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Args:
            x: (N, 4) tensor — RGB + exposure per pixel

        Returns:
            (N, 5) tensor — log-space skin parameters
        """
        x = self.act(self.fc_enc_in(x))
        x = self.act(self.fc_enc(x))
        return self.act_final(self.fc_enc_out(x))

In [ ]:
class BioSkinDecoder(torch.nn.Module):
    def __init__(self, checkpoint_path: str, device: str = 'cpu'):
        super().__init__()
        self.checkpoint_path = checkpoint_path
        self.fc_dec_in = nn.Linear(6,512)
        self.fc_dec = nn.Linear(512,512)
        self.fc_dec_out = nn.Linear(512, 310)
        self.act        = nn.Tanh()
        self.act_final  = nn.Sigmoid()
        
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

        dec_weights = {
            k.replace("module.", ""): v
            for k, v in cktp.items()
            if "fc_dec" in k
        } 
        
        missing, unexpected = self.load_state_dict(dec_weights, strict=True)
        if missing:
            raise RuntimeError(f"Missing keys in checkpoint: {missing}")
        if unexpected:
            print(f"  [BioSkinDecoder] Unexpected keys (ignored): {unexpected}")
        
        self.eval()
        self.requires_grad_(False)
        self.to(device)
        self.device = device

        print(f"[BioSkinDecoder] ✓ Loaded on {device}")
        print(f"[BioSkinDecoder]   fc_dec_in:  {self.fc_dec_in.weight.shape}")
        print(f"[BioSkinDecoder]   fc_dec:     {self.fc_dec.weight.shape}")
        print(f"[BioSkinDecoder]   fc_dec_out: {self.fc_dec_out.weight.shape}")
        
    

    def forward(self, x):
        x = self.act(self.fc_dec_in(x))
        x = self.act(self.fc_dec(x))       
        return self.act_final(self.fc_dec_out(x))

In [20]:
def _load_image(self, albedo_path: str) -> np.ndarray:
    """
    Load image exactly matching BioSkin's bio_io.load_image pipeline:
    1. cv2.imread → BGR float32 / 255
    2. sRGB_to_linear → always applies power branch (BioSkin bug replicated)
    3. clip to [0, 1]
    Shared by: extract_stats_from_image, process_image
    """
    import cv2
    image = cv2.imread(albedo_path, cv2.IMREAD_ANYCOLOR | cv2.IMREAD_ANYDEPTH)
    image = image.astype(np.float32) / 255.0
    # Replicate BioSkin's sRGB_to_linear exactly:
    # their any() check always returns False so power branch always runs
    image = ((image + 0.055) / 1.055) ** 2.4
    image = np.clip(image, 0, 1)
    return image

def _tensor_to_numpy(self, tensor: torch.Tensor) -> np.ndarray:
    """
    Convert any-shape tensor to (H, W, 3) float32 numpy in [0, 1].
    Accepted shapes: (B,C,H,W), (C,H,W), (H,W,C).
    Shared by: extract_stats_from_tensor
    """
    if isinstance(tensor, torch.Tensor):
        t = tensor.detach().cpu().float()
    else:
        t = torch.from_numpy(np.array(tensor, dtype=np.float32))

    if t.dim() == 4:
        t = t[0]
    if t.dim() == 3 and t.shape[0] in (1, 3):
        t = t.permute(1, 2, 0)
    if t.shape[-1] == 1:
        t = t.repeat(1, 1, 3)

    srgb = t.numpy().astype(np.float32)
    linear = np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)
    return linear.astype(np.float32)

def _infer_param_maps(self, img_np: np.ndarray) -> dict:
    """
    Run encoder on a (H, W, 3) float32 numpy array.
    Returns {param_name: (H, W) float32 array} in physical space.

    Key fix: appends DEFAULT_EXPOSURE to each pixel before encoder
    because BioSkinAO.json has exposure_aware: true → input is 4D not 3D.

    Shared by: extract_stats_from_image, extract_stats_from_tensor
    """
    H, W    = img_np.shape[:2]
    N       = H * W

    # Flatten pixels to (N, 3)
    pixels_rgb = img_np.reshape(N, 3)

    # Append exposure channel → (N, 4)
    pixels = pixels_rgb

    # Run encoder in batches on GPU
    all_params = []
    tensor_in  = torch.from_numpy(pixels).to(self.device)

    with torch.no_grad():
        for start in range(0, N, self.batch_size):
            end   = min(start + self.batch_size, N)
            batch = tensor_in[start:end]                  # (B, 4)
            out   = self.encoder(batch).cpu().numpy()     # (B, 5) log-space
            all_params.append(out)

    raw_params = np.concatenate(all_params, axis=0)       # (N, 5) log-space

    # Convert log-space → physical values
    physical = raw_params[:, :5].reshape(H, W, 5)

    return {
        'melanin':             physical[:, :, 0],
        'hemoglobin':          physical[:, :, 1],
        'epidermal_thickness': physical[:, :, 2],
        'eumelanin_ratio':     physical[:, :, 3],
        'oxygenation':         physical[:, :, 4],
        # channel 5 exists in checkpoint but unused per paper
    }

def _compute_full_stats(self, param_maps: dict):
    """
    Compute statistics + partial condition vector.
    Returns (stats_dict, partial_c_ndarray).
    Shared by: extract_stats_from_image, extract_stats_from_tensor
    """
    stats     = self._compute_statistics(param_maps)
    partial_c = self._build_partial_condition_vector(stats)
    return stats, partial_c

def _compute_statistics(self, param_maps: dict) -> dict:
    """
    mode/std per parameter (paper Section 3.1).
    melanin + hemoglobin → mode AND std. Others → mode only.
    """
    stats = {}
    for name in self.param_names:
        if name not in param_maps:
            continue
        pmap = param_maps[name]
        flat = pmap.flatten()
        flat = flat[~np.isnan(flat) & np.isfinite(flat)]
        mode_val = self._compute_mode(flat)

        if name in self.params_with_std:
            stats[name] = {
                'mode': float(mode_val),
                'std':  float(np.std(flat)) if len(flat) > 0 else 0.0,
            }
        else:
            stats[name] = {
                'mode': float(mode_val),
            }
    return stats

In [ ]:
Class BioSkinFull(nn.Module):
    def __init__(self, checkpoint_path, device='cuda',batch_size= 65536):
        super().__init__()
        
        # Shared config
        self.param_names     = PARAM_NAMES
        self.param_notation  = PARAM_NOTATION
        self.params_with_std = PARAMS_WITH_STD
        self.param_colormaps = PARAM_COLORMAPS
        self.batch_size      = batch_size
        self.device          = device
                 
        self.encoder = BioSkinEncoder(checkpoint_path, device)
        self.decoder = BioSkinDecoder(checkpoint_path, device)
        self.device = device
    
      

    
    
    def forward(self, x):
        skin_params = self.encoder(x)
        reconstructed_rgb = self.decoder(skin_params)
        return skin_params, reconstructed_rgb

In [21]:
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the MIT license found in the
# LICENSE file in the root directory of this source tree.


import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = 'true'
import cv2
import torch
import numpy as np
import imageio
import re
import bioskin.spectrum.color_spectrum as color


def get_file_list(input_folder):
    valid_extensions = ['.exr', '.jpeg', '.jpg', '.png']
    filename_list = os.listdir(input_folder)
    # remove masks from the list (strings containing "_mask")
    filename_list = [filename for filename in filename_list if "_mask" not in filename]
    filename_list = [filename for filename in filename_list if "_spec" not in filename]

    filename_list_no_ext = []
    extensions = []
    # get names and extensions
    for i in range(0, len(filename_list)):
        extension = os.path.splitext(filename_list[i])[1]
        if extension in valid_extensions:
            filename_list_no_ext.append(os.path.splitext(filename_list[i])[0])
            extensions.append(extension)

    if '.DS_Store' in filename_list_no_ext:  # This is for Mac OS users
        index_to_delete = filename_list_no_ext.index('.DS_Store')
        if index_to_delete != -1:
            del extensions[index_to_delete]
        filename_list_no_ext.remove('.DS_Store')
    return filename_list_no_ext, extensions


def vectorize_image(input_image, device, monochrome=False):
    row, col, channel = input_image.shape
    # reshape to nx3
    input_image_vec = input_image.reshape((row * col, 3))
    if monochrome == 1:
        input_image_vec = input_image_vec[:, 0]
    input_image_vec = np.clip(input_image_vec, a_min=0.0, a_max=1.0)  # clamping to 0 1
    tensor = torch.from_numpy(input_image_vec.astype("float32"))
    tensor = tensor.to(device=device)
    return tensor


def replace_nan_with_valid_neighbor(arr):
    valid_neighbors = arr[~np.isnan(arr)]
    if valid_neighbors.size > 0:
        return valid_neighbors[0]
    else:
        return np.nan


def detect_and_fix_nans(image, path_to_image=""):
    nan_indices = np.argwhere(np.isnan(image))
    for index in nan_indices:
        print("Warning: fixing NaNs found in loaded image " + path_to_image)
        i, j, k = index
        iminus = max(0, i - 1)
        iplus = min(image.shape[0] - 1, i + 1)
        jminus = max(0, j - 1)
        jplus = min(image.shape[1] - 1, j + 1)
        neighbors = [(iminus, j), (iplus, j), (i, jminus), (i, jplus)]
        for neighbor_i, neighbor_j in neighbors:
            if not np.isnan(image[neighbor_i, neighbor_j, :].any()):
                image[i, j, :] = image[neighbor_i, neighbor_j, :]
                break
    return image


def load_image(path_to_image, max_width=0, verbose=True):
    try:
        image = cv2.imread(path_to_image, cv2.IMREAD_ANYCOLOR | cv2.IMREAD_ANYDEPTH)
        if image is None:
            raise FileNotFoundError
    except FileNotFoundError:
        print(f"File {path_to_image} not found.")
        return None
    extension = os.path.splitext(path_to_image)[1]

    if verbose:
        print(path_to_image)
        print("Image size: ", image.shape)

    # If png, apply gamma and normalize
    if extension == ".png" or extension == ".jpg" or extension == ".jpeg":
        image = image.astype(np.float32) / 255.0
        image = color.sRGB_to_linear(image)
        image = np.clip(image, 0, 1)

    image = detect_and_fix_nans(image, path_to_image)

    if image.shape[1] > max_width != 0:
        ratio = max_width / image.shape[1]
        new_width = max_width
        new_height = int(image.shape[0] * ratio)
        image = cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_AREA)
        if verbose:
            print("Downsampling image " + path_to_image + " to " + str(max_width))
    return image


def save_image(image, path_to_save, max_width=0, verbose=True):
    try:
        if image is None:
            raise ValueError("Image is None.")
    except ValueError as e:
        print(e)
        return None
    extension = os.path.splitext(path_to_save)[1]
    # If png, apply gamma and normalize
    if extension == ".png" or extension == ".jpg" or extension == ".jpeg":
        image = np.clip(image, 0, 1)
        image = color.linear_to_sRGB(image)
        image = (image * 255.0).astype(np.uint8)
    if image.shape[1] > max_width != 0:
        ratio = max_width / image.shape[1]
        new_width = max_width
        new_height = int(image.shape[0] * ratio)
        image = cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_AREA)
        if verbose:
            print("Downsampling image to " + str(max_width))
    # Save the image
    cv2.imwrite(path_to_save, image)
    if verbose:
        print(f"Image saved at {path_to_save}")


def vector1D_to_image(vector, row, col, channels=3):
    return vector.reshape((row, col, channels)).astype("float32")


def save_jpeg(path, image, linear_input=True):
    if linear_input:
        image = color.linear_to_sRGB(image)
    cv2.imwrite(path + '.jpeg', image * 255)


def tensor_to_image(t, shape, channels=3):
    t_image = vector1D_to_image(t.cpu().detach().numpy(), shape[0], shape[1], channels=channels)
    return t_image


def cpu_tensor_to_image(t, shape, channels=3):
    t_image = vector1D_to_image(t.numpy(), shape[0], shape[1], channels=channels)
    return t_image


def save_tensor_to_image(path, t, shape, channels=3, cpu=False):
    if cpu:
        t_image = cpu_tensor_to_image(t, shape, channels)
    else:
        t_image = tensor_to_image(t, shape, channels)
    cv2.imwrite(path + ".exr", t_image)
    save_jpeg(path, t_image)


def extract_number(filename):
    return int(re.search(r'(\d+)', filename).group(1))


def create_gif_from_images(folder_path, gif_name, image_extension="jpg", duration=0.1, max_frames=None):
    image_files = [f for f in os.listdir(folder_path) if f.endswith(image_extension)]

    # Sort files based on the numerical value embedded in the filename
    sorted_files = sorted(image_files, key=extract_number)

    # Limit the number of frames if max_frames is specified
    if max_frames is not None:
        sorted_files = sorted_files[:max_frames]

    images = [imageio.imread(os.path.join(folder_path, f)) for f in sorted_files]

    # Create GIF with controlled speed (frame duration)
    imageio.mimsave(os.path.join(folder_path, gif_name), images, duration=duration, loop=0)